# Noise Sensitivity

This notebook studies how additive response noise affects recovery of the active direction in

$$u=\sin(x+y)+\varepsilon.$$

The true direction is $\theta/\pi=0.25$. MI-DL and SI-DL are compared at seven noise levels using 70 independent random repeats.

The calculation is written as a linear notebook workflow. It contains no helper functions and no `main()` entry point.


## 1. Imports and project paths

Load the local MI-DL and SI-DL implementations together with the numerical, optimization, and plotting libraries.


In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import differential_evolution
from sklearn.model_selection import train_test_split

OUTPUT_DIR = Path.cwd()
CSV_DIR = OUTPUT_DIR / "csv"
PROJECT_ROOT = OUTPUT_DIR.parents[1]
os.environ.setdefault("MPLCONFIGDIR", str(PROJECT_ROOT / ".matplotlib-cache"))

for project_path in (PROJECT_ROOT / "Compare" / "MI-DL", PROJECT_ROOT / "SI-DL-main"):
    if str(project_path) not in sys.path:
        sys.path.insert(0, str(project_path))

import midl
import SI_DL


## 2. Experiment settings

Every repeat uses the same base coordinates and standard-normal noise realization across all seven noise levels. Only the noise amplitude changes, making comparisons within a repeat directly paired.


In [ ]:
BASE_RANDOM_STATE = 42
N_REPEATS = 70
N_SAMPLES = 500
NOISE_LEVELS_PERCENT = (0, 5, 10, 15, 20, 25, 30)
TEST_SIZE = 0.25

TRUE_THETA = np.pi / 4.0
TRUE_THETA_OVER_PI = 0.25
THETA_BOUNDS = (0.0, np.pi)

DE_MAXITER = 20
DE_POPSIZE = 8
DE_TOL = 0.001
DE_POLISH = True

MI_K_NEIGHBORS = 10
SI_FIXED_BANDWIDTH = 0.055

RESULTS_CSV = CSV_DIR / "results.csv"
SUMMARY_CSV = CSV_DIR / "summary.csv"
PLOT_PATH = OUTPUT_DIR / "noise.png"

METHOD_COLORS = {"MI-DL": "#3268a8", "SI-DL": "#d26b4f"}
CSV_DIR.mkdir(parents=True, exist_ok=True)


## 3. Run the paired noise experiment

For each repeat, draw 500 values of $x$ and $y$ uniformly on $[0,5]$ and generate a clean response. A fixed standard-normal vector is scaled by each requested percentage of the clean-response standard deviation.

Differential evolution then minimizes the MI-DL or SI-DL objective over $0\leq\theta\leq\pi$.


In [ ]:
result_records = []

for repeat_index in range(N_REPEATS):
    random_state = BASE_RANDOM_STATE + repeat_index
    rng = np.random.default_rng(random_state)

    x = rng.uniform(0.0, 5.0, size=N_SAMPLES)
    y = rng.uniform(0.0, 5.0, size=N_SAMPLES)
    u_clean = np.sin(x + y)
    standard_noise = rng.normal(0.0, 1.0, size=N_SAMPLES)

    train_idx, _ = train_test_split(
        np.arange(N_SAMPLES),
        test_size=TEST_SIZE,
        random_state=random_state,
    )
    x_train = x[train_idx]
    y_train = y[train_idx]

    for noise_percent in NOISE_LEVELS_PERCENT:
        noise_scale = (
            noise_percent / 100.0
        ) * np.std(u_clean, ddof=0)
        u_noisy = u_clean + noise_scale * standard_noise
        u_train = u_noisy[train_idx]
        error_scale = float(np.var(u_train, ddof=0))

        mi_seed = random_state + noise_percent * 101
        mi_result = differential_evolution(
            lambda parameters: float(
                midl.information_lower_bound(
                    np.cos(float(parameters[0])) * x_train
                    + np.sin(float(parameters[0])) * y_train,
                    u_train,
                    k_neighbors=MI_K_NEIGHBORS,
                    random_state=random_state,
                )["epsilon_lb"]
                / error_scale
            ),
            bounds=[THETA_BOUNDS],
            maxiter=DE_MAXITER,
            popsize=DE_POPSIZE,
            tol=DE_TOL,
            seed=mi_seed,
            polish=DE_POLISH,
            updating="immediate",
            workers=1,
        )
        mi_theta = float(mi_result.x[0])
        mi_coordinate = (
            np.cos(mi_theta) * x_train + np.sin(mi_theta) * y_train
        )
        mi_values = midl.information_lower_bound(
            mi_coordinate,
            u_train,
            k_neighbors=MI_K_NEIGHBORS,
            random_state=random_state,
        )
        mi_theta_over_pi = mi_theta / np.pi
        mi_abs_error = abs(mi_theta_over_pi - TRUE_THETA_OVER_PI)
        result_records.append({
            "repeat": repeat_index,
            "random_state": random_state,
            "noise_percent": noise_percent,
            "method": "MI-DL",
            "theta": mi_theta,
            "theta_over_pi": mi_theta_over_pi,
            "theta_abs_error": mi_abs_error,
            "normalized_error": mi_abs_error / TRUE_THETA_OVER_PI,
            "objective_error": float(mi_values["epsilon_lb"] / error_scale),
            "mutual_information": float(mi_values["mutual_information"]),
            "si_score": np.nan,
        })

        si_seed = random_state + noise_percent * 101 + 10_000
        si_result = differential_evolution(
            lambda parameters: float(
                1.0
                - SI_DL.explained_variance_score(
                    np.cos(float(parameters[0])) * x_train
                    + np.sin(float(parameters[0])) * y_train,
                    u_train,
                    bandwidth=SI_FIXED_BANDWIDTH,
                )["S_cov"]
            ),
            bounds=[THETA_BOUNDS],
            maxiter=DE_MAXITER,
            popsize=DE_POPSIZE,
            tol=DE_TOL,
            seed=si_seed,
            polish=DE_POLISH,
            updating="immediate",
            workers=1,
        )
        si_theta = float(si_result.x[0])
        si_coordinate = (
            np.cos(si_theta) * x_train + np.sin(si_theta) * y_train
        )
        si_values = SI_DL.explained_variance_score(
            si_coordinate,
            u_train,
            bandwidth=SI_FIXED_BANDWIDTH,
        )
        si_theta_over_pi = si_theta / np.pi
        si_abs_error = abs(si_theta_over_pi - TRUE_THETA_OVER_PI)
        result_records.append({
            "repeat": repeat_index,
            "random_state": random_state,
            "noise_percent": noise_percent,
            "method": "SI-DL",
            "theta": si_theta,
            "theta_over_pi": si_theta_over_pi,
            "theta_abs_error": si_abs_error,
            "normalized_error": si_abs_error / TRUE_THETA_OVER_PI,
            "objective_error": float(1.0 - si_values["S_cov"]),
            "mutual_information": np.nan,
            "si_score": float(si_values["S_cov"]),
        })

    print(f"Experiment repeat {repeat_index + 1}/{N_REPEATS}")

results = pd.DataFrame(result_records)
results.to_csv(RESULTS_CSV, index=False)
print(f"Saved {len(results)} rows to {RESULTS_CSV.relative_to(OUTPUT_DIR)}")

summary_records = []

for (method, noise_percent), group in results.groupby(
    ["method", "noise_percent"],
    sort=False,
):
    direction_error = group["normalized_error"].to_numpy(float)
    summary_records.append({
        "method": method,
        "noise_percent": int(noise_percent),
        "n_repeats": int(group["repeat"].nunique()),
        "error_q25": float(np.quantile(direction_error, 0.25)),
        "error_median": float(np.median(direction_error)),
        "error_q75": float(np.quantile(direction_error, 0.75)),
        "theta_median": float(group["theta_over_pi"].median()),
        "objective_median": float(group["objective_error"].median()),
    })

summary = pd.DataFrame(summary_records)
summary.to_csv(SUMMARY_CSV, index=False)
summary


## 4. Plot noise sensitivity

The lines show the median normalized direction error across 70 repeats. Shaded regions indicate the 25th–75th percentile interval. This is the only figure retained in the noise example.


In [ ]:
fig, ax = plt.subplots(figsize=(7.4, 4.8))
positive_values = summary.loc[summary["error_q25"] > 0.0, "error_q25"]
plot_floor = float(positive_values.min() * 0.2) if not positive_values.empty else 1.0e-8

for method, group in summary.groupby("method", sort=False):
    group = group.sort_values("noise_percent")
    x_values = group["noise_percent"].to_numpy(float)
    median = np.maximum(group["error_median"].to_numpy(float), plot_floor)
    lower = np.maximum(group["error_q25"].to_numpy(float), plot_floor)
    upper = np.maximum(group["error_q75"].to_numpy(float), plot_floor * 1.01)

    ax.plot(
        x_values,
        median,
        marker="o",
        markersize=6.5,
        linewidth=2.4,
        color=METHOD_COLORS[method],
        label=method,
    )
    ax.fill_between(
        x_values,
        lower,
        upper,
        color=METHOD_COLORS[method],
        alpha=0.18,
        label=f"{method} 25%-75%",
    )

ax.set_yscale("log")
ax.set_xlim(min(NOISE_LEVELS_PERCENT) - 1.5, max(NOISE_LEVELS_PERCENT) + 1.5)
ax.set_xticks(NOISE_LEVELS_PERCENT)
ax.set_xlabel("Noise Level (%)", fontsize=15)
ax.set_ylabel("Normalized Direction Error", fontsize=15)
ax.tick_params(axis="both", which="major", labelsize=13)
ax.tick_params(axis="both", which="minor", labelsize=11)
ax.legend(frameon=False, fontsize=11)
fig.tight_layout()
fig.savefig(PLOT_PATH, dpi=240)
plt.show()

print(f"Saved {PLOT_PATH.name}")


## 5. Output files

The experiment produces one figure and two compact CSV tables. No per-repeat sample files or secondary figures are retained.


In [ ]:
pd.DataFrame({
    "file": [
        str(RESULTS_CSV.relative_to(OUTPUT_DIR)),
        str(SUMMARY_CSV.relative_to(OUTPUT_DIR)),
        PLOT_PATH.name,
    ],
    "description": [
        "All optimized directions from 70 repeats",
        "Median and interquartile summaries",
        "Final noise-sensitivity comparison",
    ],
})
